# 01 — NMF global das frases (modelo final)

Primeiro estágio do modelo final de sentenças: NMF (scikit-learn, TF-IDF) sobre a **frase
central** de todas as frases filtradas, com K fixo em `params.yaml::final_sentence_pipeline.global_k`.

- **Entrada:** `02-sentences/data/output/youtube_sent/youtube_sent_*/corpus_sentencas.csv` (o mais recente).
- **Saída:** `03-topic-modeling/data/output/youtube_sent/nmf_global/nmf_global_sentence_k{K}_{data}/`
  com `nmf_results.csv`, `topics.csv`, `topics.json`, `topic_weights.npz`, `model.pkl` e `manifest.json`.
- **Depois deste notebook:** ler os temas, escolher os temas de aspecto e preencher
  `global_run`, `selected_topics` e `expected_sentences` no `params.yaml` (última seção).
  Em seguida, rodar `02_nmf_topicos.ipynb`.

O código do modelo está em `03-topic-modeling/scripts/run_global_nmf_sentences.py`; este
notebook só o chama e mostra os resultados. Os pesos são ativações NMF normalizadas por frase,
não probabilidades calibradas.

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "03-topic-modeling").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "03-topic-modeling" / "scripts"))
sys.path.insert(0, str(ROOT / "03-topic-modeling" / "notebooks"))
pd.set_option("display.max_colwidth", 200)

PARAMS_PATH = ROOT / "03-topic-modeling" / "configs" / "params.yaml"
params = yaml.safe_load(PARAMS_PATH.read_text(encoding="utf-8"))
FINAL = params["final_sentence_pipeline"]
SEED = params["seed"]

import run_global_nmf_sentences as nmf_global

K = FINAL["global_k"]
print("K global:", K, "| seed:", SEED)

## 1. Treinar

`RUN_EXISTENTE = None` treina um run novo. Para só reabrir um run já treinado, coloque o nome
da pasta (ex.: `"nmf_global_sentence_k20_20261006_101500"`).

In [ ]:
RUN_EXISTENTE = None

if RUN_EXISTENTE:
    RUN = nmf_global.OUTPUT_ROOT / RUN_EXISTENTE
else:
    fonte = nmf_global.source_file()
    print("Fonte:", fonte)
    RUN = nmf_global.run(fonte, [K], SEED, max_features=12000, text_column="sentence")[0]
print("Run:", RUN.name)

In [ ]:
manifest = json.loads((RUN / "manifest.json").read_text(encoding="utf-8"))
pd.Series(manifest).to_frame("valor")

## 2. Temas: termos e tamanho

In [ ]:
topics = pd.read_csv(RUN / "topics.csv")
res = pd.read_csv(RUN / "nmf_results.csv", dtype={"sent_id": str, "post_id": str})
videos = res.loc[res.topic_id >= 0].groupby("topic_id").post_id.nunique().rename("n_videos")
topics = topics.join(videos, on="topic_id")
topics["pct_frases"] = (100 * topics.n_sentences / len(res)).round(1)
print(f"{len(res)} frases; {int((res.topic_id == -1).sum())} sem termos no vocabulário (tema -1)")
topics[["topic_id", "n_sentences", "pct_frases", "n_videos", "top_terms"]]

## 3. Temas por categoria de produto

Percentual das frases de cada tema por categoria. Compare com a linha **base** (todas as
frases): temas de aspecto compartilhado acompanham a base; temas de categoria se concentram.
Lembre que canal e categoria se confundem neste corpus (ver idealização §5.3C).

In [ ]:
validas = res.loc[res.topic_id >= 0]
tab = pd.crosstab(validas.topic_id, validas.product_category, normalize="index").mul(100)
base = validas.product_category.value_counts(normalize=True).mul(100).rename("base")
cols = base.index[:6]
pd.concat([tab[cols], base[cols].to_frame().T]).round(0)

## 4. Frases exemplares

Para cada tema: frases de maior peso (vídeos distintos) e frases sorteadas. Leia as duas
listas: o peso alto tende a favorecer frases curtas e descritivas.

In [ ]:
exemplos = json.loads((RUN / "topics.json").read_text(encoding="utf-8"))
for t in exemplos:
    print(f"\n=== Tema {t['topic_id']} ({t['n_sentences']} frases) — {t['top_terms'][:110]}")
    print("  maior peso:")
    for e in t["strong_examples"][:4]:
        print(f"    [{e['weight']:.2f}|{e['category']}] {e['sentence'][:150]}")
    print("  sorteadas:")
    for e in t["random_examples"][:3]:
        print(f"    [{e['weight']:.2f}|{e['category']}] {e['sentence'][:150]}")

## 5. Escolher os temas de aspecto e fixar no `params.yaml`

A escolha é qualitativa: temas que descrevem aspectos do produto (bateria, tela, teclado...),
não temas de fala genérica. Liste os IDs em `TEMAS` e copie o bloco impresso para
`params.yaml::final_sentence_pipeline`. Os IDs só valem para este run.

In [ ]:
TEMAS = []   # ex.: [1, 3, 5, 7, 9, 10, 11, 14, 15, 17]

n_sel = int(res.topic_id.isin(TEMAS).sum())
print(f"{len(TEMAS)} temas, {n_sel} frases ({100 * n_sel / len(res):.1f}% das frases)\n")
print("Copie para params.yaml::final_sentence_pipeline:")
print(f"  global_run: {RUN.name}")
print(f"  selected_topics: {sorted(TEMAS)}")
print(f"  expected_sentences: {n_sel}")